# Chapter 10 — Neural Learning About Edges and Corners: Intro to Convolutional Neural Networks

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning)
**Chapter 10:** Neural Learning About Edges and Corners: Intro to Convolutional Neural Networks

Notebook ini berisi reproduksi kode dari buku (diambil dari repository
resmi penulis) beserta penjelasan konsep dan rangkuman chapter.

## Pendahuluan

Chapter 10 memperkenalkan **convolutional neural network (CNN)**, ide
tentang **menggunakan kembali weights di banyak tempat** ("structure
trick"). Contoh: mendeteksi garis dan tepi pada gambar MNIST.

### Menggunakan weights yang sama di banyak tempat

Jika jaringan perlu mengenali "ide" yang sama (misal sebuah garis) di
bagian gambar yang berbeda, memakai weights terpisah untuk tiap bagian
berarti tiap bagian harus belajar sendiri. **Structure trick:** pakai
weights yang sama di semua tempat tersebut agar weights itu belajar dari
lebih banyak data dan menjadi lebih cerdas.

### Lapisan konvolusi (convolutional layer)

Bayangkan banyak **layer linear kecil** (misal kernel 3x3) yang bergeser
(*sliding*) di seluruh gambar. Setiap kernel menghasilkan output untuk
satu patch, dan output seluruh patch digabung. Setiap kernel mempelajari
satu fitur (garis, sudut, bentuk) dan hasilnya diteruskan ke layer
berikutnya. Implementasi NumPy memakai fungsi pembantu `get_image_section`
untuk mengambil patch gambar, lalu `sects` ditumpuk menjadi input layer
berikutnya.

# Upgrading our MNIST Network

**Jaringan MNIST dengan konvolusi:** gambar 28x28 diambil dalam patch
3x3; setiap patch dimasukkan ke kumpulan kernel yang sama
(`kernels`, bentuk 9 x `num_kernels`). Output setiap patch
digabung menjadi satu vektor yang menjadi input layer berikutnya (softmax).
Dengan weights yang dipakai ulang di seluruh gambar, jaringan memiliki
lebih sedikit parameter dan generalisasinya lebih baik, sehingga akurasi
uji MNIST meningkat dibanding jaringan penuh (fully connected).

In [ ]:
import numpy as np, sys
np.random.seed(1)

from keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

images, labels = (x_train[0:1000].reshape(1000,28*28) / 255,
                  y_train[0:1000])


one_hot_labels = np.zeros((len(labels),10))
for i,l in enumerate(labels):
    one_hot_labels[i][l] = 1
labels = one_hot_labels

test_images = x_test.reshape(len(x_test),28*28) / 255
test_labels = np.zeros((len(y_test),10))
for i,l in enumerate(y_test):
    test_labels[i][l] = 1

def tanh(x):
    return np.tanh(x)

def tanh2deriv(output):
    return 1 - (output ** 2)

def softmax(x):
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)

alpha, iterations = (2, 300)
pixels_per_image, num_labels = (784, 10)
batch_size = 128

input_rows = 28
input_cols = 28

kernel_rows = 3
kernel_cols = 3
num_kernels = 16

hidden_size = ((input_rows - kernel_rows) * 
               (input_cols - kernel_cols)) * num_kernels

# weights_0_1 = 0.02*np.random.random((pixels_per_image,hidden_size))-0.01
kernels = 0.02*np.random.random((kernel_rows*kernel_cols,
                                 num_kernels))-0.01

weights_1_2 = 0.2*np.random.random((hidden_size,
                                    num_labels)) - 0.1



def get_image_section(layer,row_from, row_to, col_from, col_to):
    section = layer[:,row_from:row_to,col_from:col_to]
    return section.reshape(-1,1,row_to-row_from, col_to-col_from)

for j in range(iterations):
    correct_cnt = 0
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end=((i * batch_size),((i+1)*batch_size))
        layer_0 = images[batch_start:batch_end]
        layer_0 = layer_0.reshape(layer_0.shape[0],28,28)
        layer_0.shape

        sects = list()
        for row_start in range(layer_0.shape[1]-kernel_rows):
            for col_start in range(layer_0.shape[2] - kernel_cols):
                sect = get_image_section(layer_0,
                                         row_start,
                                         row_start+kernel_rows,
                                         col_start,
                                         col_start+kernel_cols)
                sects.append(sect)

        expanded_input = np.concatenate(sects,axis=1)
        es = expanded_input.shape
        flattened_input = expanded_input.reshape(es[0]*es[1],-1)

        kernel_output = flattened_input.dot(kernels)
        layer_1 = tanh(kernel_output.reshape(es[0],-1))
        dropout_mask = np.random.randint(2,size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        layer_2 = softmax(np.dot(layer_1,weights_1_2))

        for k in range(batch_size):
            labelset = labels[batch_start+k:batch_start+k+1]
            _inc = int(np.argmax(layer_2[k:k+1]) == 
                               np.argmax(labelset))
            correct_cnt += _inc

        layer_2_delta = (labels[batch_start:batch_end]-layer_2)\
                        / (batch_size * layer_2.shape[0])
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * \
                        tanh2deriv(layer_1)
        layer_1_delta *= dropout_mask
        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        l1d_reshape = layer_1_delta.reshape(kernel_output.shape)
        k_update = flattened_input.T.dot(l1d_reshape)
        kernels -= alpha * k_update
    
    test_correct_cnt = 0

    for i in range(len(test_images)):

        layer_0 = test_images[i:i+1]
#         layer_1 = tanh(np.dot(layer_0,weights_0_1))
        layer_0 = layer_0.reshape(layer_0.shape[0],28,28)
        layer_0.shape

        sects = list()
        for row_start in range(layer_0.shape[1]-kernel_rows):
            for col_start in range(layer_0.shape[2] - kernel_cols):
                sect = get_image_section(layer_0,
                                         row_start,
                                         row_start+kernel_rows,
                                         col_start,
                                         col_start+kernel_cols)
                sects.append(sect)

        expanded_input = np.concatenate(sects,axis=1)
        es = expanded_input.shape
        flattened_input = expanded_input.reshape(es[0]*es[1],-1)

        kernel_output = flattened_input.dot(kernels)
        layer_1 = tanh(kernel_output.reshape(es[0],-1))
        layer_2 = np.dot(layer_1,weights_1_2)

        test_correct_cnt += int(np.argmax(layer_2) == 
                                np.argmax(test_labels[i:i+1]))
    if(j % 1 == 0):
        sys.stdout.write("\n"+ \
         "I:" + str(j) + \
         " Test-Acc:"+str(test_correct_cnt/float(len(test_images)))+\
         " Train-Acc:" + str(correct_cnt/float(len(images))))

## Key Ideas

- Weights yang dipakai ulang membuat jaringan lebih cerdas dan lebih ringkas.
- Convolutional layer menggeser kernel kecil di seluruh gambar.
- Tiap kernel mempelajari satu fitur (garis, sudut, bentuk).
- Structure trick: gunakan weights yang sama untuk ide yang sama di banyak tempat.

## Kesimpulan


Chapter 10 menegaskan bahwa **menggunakan kembali weights** adalah salah
satu inovasi terpenting deep learning. Convolutional layer menerapkan
kernel kecil yang sama pada banyak posisi gambar, sehingga fitur
level rendah (garis, tepi, sudut) dipelajari sekali dan dipakai di mana
saja, lalu digabung menjadi fitur tingkat tinggi (misal kucing).
**Structure trick:** jika sebuah ide perlu dipakai di banyak tempat, pakai
weights yang sama di tempat-tempat itu.